# AA-510 Agentic AI Systems (Group 4)

Team Member: Jeffery Smith、Jinyuan He、Mostafa Zamaniturk

**Part A: Agent Deliverable - Technical Artifacts**
- A notebook that contains your data pipeline (DE is the owner)
- A notebook that shows your agent definition (AIE is the owner)
- Review the rubric to determine agent requirements.
- An example of five traces, including the evaluation of the agent (AIE is the owner)
- Traces should be presented via an established provider. Databricks will leverage MLflow by default; alternatives are Arize Phoenix, LangSmith, and Braintrust, among others.
- There should be at least 2 different LLMs used within the agent on the same trace to compare performance (this counts as one trace per the five above).
- Evaluation can be an LLM judge (strongly encouraged) or manual.
- Provide commentary in a notebook cell that talks about the agent performance, and what the evaluation showed, including a comparison between the agent using different LLMs.
- Show 2 examples of the agent gracefully rejecting an irrelevant user input.

# 1. Project Overview

### 1.1 Business problem
### 1.2 Agent goal
### 1.3 Team roles: PM / DE / AIE

# 2. Data Pipeline

### 2.1 Load dataset

In [0]:
%pip install -U -qqqq kaggle backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp nest_asyncio "gepa>=0.1.0" "databricks-vectorsearch<0.74"
%pip install -U  sentence-transformers 
dbutils.library.restartPython()

In [0]:
%pip install --upgrade databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%restart_python

In [0]:
# Download dataset from Kaggle
import os
import pandas as pd

# Create directory if it doesn't exist
os.makedirs('/tmp/ai_reviews', exist_ok=True)

# Download to local tmp directory
!kaggle datasets download -d jahnavikachhia23/the-generative-ai-ecosystem-50k-user-reviews-2026 --unzip -p /tmp/ai_reviews

# Load the CSV into pandas first
pandas_df = pd.read_csv('/tmp/ai_reviews/The_ Generative_AI_Ecosystem_50k_User_Reviews_2026.csv')

# Convert to Spark DataFrame
df = spark.createDataFrame(pandas_df)

display(df)

In [0]:
# Create new catalog "group4"
spark.sql("CREATE CATALOG IF NOT EXISTS group4")

# Save the dataset as a table in the new catalog (using full 3-part name)
df.write.format("delta").mode("overwrite").saveAsTable("group4.default.ai_reviews")

### 2.2 Exploratory Data Analysis

In [0]:
# Exploratory Data Analysis (EDA) on ai_reviews dataset

# Show schema
df.printSchema()

# Show summary statistics for numeric columns
display(df.describe())

# Show first 10 rows
display(df.limit(10))

# Count total rows
row_count = df.count()

# Show distribution of review ratings (if column exists)
if 'rating' in df.columns:
    display(df.groupBy('rating').count().orderBy('rating'))

# Show top 10 most frequent products (if column exists)
if 'product_name' in df.columns:
    display(df.groupBy('product_name').count().orderBy('count', ascending=False).limit(10))

# Show null counts per column
from pyspark.sql.functions import col, sum as spark_sum
null_counts = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

In [0]:
# Aggregate Table - Reviews by Product Sentiment Categories
from pyspark.sql.functions import when, col

# Load the base table
df = spark.table("group4.default.ai_reviews")

# Create sentiment categories based on Sentiment_Polarity
# Positive: > 0.3, Neutral: -0.3 to 0.3, Negative: < -0.3
df_with_sentiment = df.withColumn(
    "sentiment_category",
    when(col("Sentiment_Polarity") > 0.3, "Positive")
    .when(col("Sentiment_Polarity") < -0.3, "Negative")
    .otherwise("Neutral")
)

# Show distribution of sentiment categories
print("Sentiment Category Distribution:")
display(df_with_sentiment.groupBy("sentiment_category").count().orderBy("count", ascending=False))

In [0]:
# Feature Engineering: Temporal Features
from pyspark.sql.functions import to_timestamp, year, month, dayofmonth, dayofweek, hour, quarter, weekofyear

# Parse Review_Date and extract temporal features
df_temporal = df_with_sentiment.withColumn(
    "review_timestamp", to_timestamp(col("Review_Date"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "review_year", year(col("review_timestamp"))
).withColumn(
    "review_month", month(col("review_timestamp"))
).withColumn(
    "review_day", dayofmonth(col("review_timestamp"))
).withColumn(
    "review_dayofweek", dayofweek(col("review_timestamp"))  # 1=Sunday, 7=Saturday
).withColumn(
    "review_hour", hour(col("review_timestamp"))
).withColumn(
    "review_quarter", quarter(col("review_timestamp"))
).withColumn(
    "review_week", weekofyear(col("review_timestamp"))
)

# Show sample with temporal features
print("Sample data with temporal features:")
display(df_temporal.select(
    "App", "Review_Date", "review_year", "review_month", "review_day", 
    "review_dayofweek", "review_hour", "Star_Rating"
).limit(10))

In [0]:
# Aggregate Table: Reviews by Product (App)
from pyspark.sql.functions import count, avg, sum as spark_sum, min as spark_min, max as spark_max

# Aggregate by product (App)
df_product_agg = df_temporal.groupBy("App").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    spark_min("review_timestamp").alias("first_review_date"),
    spark_max("review_timestamp").alias("latest_review_date"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews"),
    count(when(col("sentiment_category") == "Neutral", 1)).alias("neutral_reviews")
).orderBy("total_reviews", ascending=False)

print("Reviews aggregated by Product (App):")
display(df_product_agg)

In [0]:
# Aggregate Table: Reviews by Date
from pyspark.sql.functions import date_format

# Create date column (without time)
df_with_date = df_temporal.withColumn(
    "review_date_only", date_format(col("review_timestamp"), "yyyy-MM-dd")
)

# Aggregate by date
df_date_agg = df_with_date.groupBy("review_date_only").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews")
).orderBy("review_date_only", ascending=False)

print("Reviews aggregated by Date:")
display(df_date_agg.limit(20))

# Save aggregate table
df_date_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_date")
print("\nAggregate table saved to group4.default.reviews_by_date")

In [0]:
# Aggregate by star rating
df_rating_agg = df_temporal.groupBy("Star_Rating").agg(
    count("*").alias("total_reviews"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    avg("Review_Length_Chars").alias("avg_review_length"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up")
).orderBy("Star_Rating")

print("Reviews aggregated by Star Rating:")
display(df_rating_agg)

# Save aggregate table
df_rating_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_rating")
print("\nAggregate table saved to group4.default.reviews_by_rating")

In [0]:
# Product Category Hierarchy
from pyspark.sql.functions import lit

# Create product category hierarchy based on App names
# Categorize AI apps into types
df_categorized = df_temporal.withColumn(
    "app_category",
    when(col("App").isin(["ChatGPT", "Claude", "Gemini", "Copilot"]), "Conversational AI")
    .when(col("App").isin(["Perplexity", "You.com"]), "Search AI")
    .when(col("App").isin(["Midjourney", "DALL-E", "Stable Diffusion"]), "Image Generation")
    .otherwise("Other AI")
).withColumn(
    "app_provider",
    when(col("App") == "ChatGPT", "OpenAI")
    .when(col("App") == "Claude", "Anthropic")
    .when(col("App") == "Gemini", "Google")
    .when(col("App") == "Copilot", "Microsoft")
    .when(col("App") == "Perplexity", "Perplexity AI")
    .otherwise("Other")
)

# Show distribution by category
print("Distribution by App Category:")
display(df_categorized.groupBy("app_category", "App").count().orderBy("app_category", "count", ascending=False))

In [0]:
# Review theme Hierarchy
from pyspark.sql.functions import lit

# Show distribution by Review_Theme
print("Distribution by App Category:")
display(df_categorized.groupBy("Review_Theme").count().orderBy("Review_Theme", "count", ascending=False))

### 2.3 Create embeddings

In [0]:

from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, ArrayType, FloatType
import pandas as pd
from sentence_transformers import SentenceTransformer

# import model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
df = spark.table("group4.default.ai_reviews")


In [0]:
# Process embeddings in batches on the driver to avoid worker download issues
# The model is already loaded in cell 19

import pandas as pd
from pyspark.sql.functions import lit, array
from pyspark.sql.types import ArrayType, FloatType

# Get all reviews
df_base = spark.table("group4.default.ai_reviews")
reviews_pdf = df_base.toPandas()

# Generate embeddings in batches
batch_size = 1000
all_embeddings = []

print(f"Processing {len(reviews_pdf)} reviews in batches of {batch_size}...")
for i in range(0, len(reviews_pdf), batch_size):
    batch = reviews_pdf["Review_Text"][i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.extend(batch_embeddings.tolist())
    if (i // batch_size + 1) % 10 == 0:
        print(f"Processed {i + len(batch)} reviews...")

print("Embeddings generated. Creating DataFrame...")

# Add embeddings to the pandas DataFrame
reviews_pdf["embedding"] = all_embeddings

# Convert back to Spark DataFrame
df_with_embeddings = spark.createDataFrame(reviews_pdf)

# Write to table
print("Writing to Delta table...")
df_with_embeddings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("group4.default.ai_reviews_embeddings")

print("✅ Table group4.default.ai_reviews_embeddings created successfully!")

# 3. Tooling Setup

### 3.1 Vector search / RAG tool

In [0]:
# RAG Vector Search Tool
# This function performs semantic similarity search and retrieves relevant reviews
# to provide context for the LLM agent's responses

from pyspark.sql.functions import col, udf, lit, struct
from pyspark.sql.types import DoubleType
import numpy as np
from sentence_transformers import SentenceTransformer
import pandas as pd

# Define cosine similarity UDF at module level (best practice for Spark)
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1_arr = np.array(vec1)
    vec2_arr = np.array(vec2)
    dot_product = np.dot(vec1_arr, vec2_arr)
    norm1 = np.linalg.norm(vec1_arr)
    norm2 = np.linalg.norm(vec2_arr)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(dot_product / (norm1 * norm2))

cosine_sim_udf = udf(cosine_similarity, DoubleType())

# Model will be loaded on first use to avoid environment sync issues
embedding_model = None

def get_embedding_model():
    """Lazy load the embedding model to avoid UDF serialization issues"""
    global embedding_model
    if embedding_model is None:
        print("Loading SentenceTransformer model...")
        embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
        print("✅ Model loaded successfully")
    return embedding_model

def vector_search(query: str, top_k: int = 10, min_similarity: float = 0.0):
    """
    Perform semantic similarity search on review embeddings.
    
    This is the core RAG retrieval function that:
    1. Converts the query to an embedding vector
    2. Computes cosine similarity with all review embeddings
    3. Returns the top-k most similar reviews
    
    Args:
        query (str): Natural language search query
        top_k (int): Number of top results to return (default: 10)
        min_similarity (float): Minimum similarity threshold 0-1 (default: 0.0)
    
    Returns:
        pd.DataFrame: Top-k most similar reviews with columns:
            - App: Application name
            - Review_Text: Full review text
            - Star_Rating: User rating (1-5)
            - Review_Date: Date of review
            - Sentiment_Polarity: Sentiment score (-1 to 1)
            - Review_Theme: Categorized theme
            - similarity_score: Cosine similarity (0-1)
    """
    # Generate query embedding
    model = get_embedding_model()
    query_embedding = model.encode(query).tolist()
    
    # Load embeddings table
    embeddings_df = spark.table("group4.default.ai_reviews_embeddings")
    
    # Compute similarity scores
    results_df = embeddings_df.withColumn(
        "similarity_score",
        cosine_sim_udf(col("embedding"), lit(query_embedding))
    )
    
    # Filter by minimum similarity and get top_k
    if min_similarity > 0:
        results_df = results_df.filter(col("similarity_score") >= min_similarity)
    
    results_df = results_df.orderBy(col("similarity_score").desc()).limit(top_k)
    
    # Select relevant columns and convert to pandas for agent consumption
    return results_df.select(
        "App",
        "Review_Text",
        "Star_Rating",
        "Review_Date",
        "Sentiment_Polarity",
        "Review_Theme",
        "similarity_score"
    ).toPandas()

def format_context_for_llm(results_df: pd.DataFrame) -> str:
    """
    Format retrieved reviews into a context string for the LLM.
    
    Args:
        results_df (pd.DataFrame): Results from vector_search()
    
    Returns:
        str: Formatted context string with review information
    """
    if len(results_df) == 0:
        return "No relevant reviews found."
    
    context_parts = []
    for idx, row in results_df.iterrows():
        context_parts.append(
            f"Review {idx+1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    
    return "\n---\n\n".join(context_parts)

# Test the vector search function
print("\n=== Testing Vector Search Tool ===")
test_query = "What do users complain about app crashes?"
print(f"Query: {test_query}\n")

test_results = vector_search(test_query, top_k=5)
print(f"Found {len(test_results)} similar reviews:\n")
display(test_results[["App", "Star_Rating", "similarity_score", "Review_Theme", "Review_Text"]])

# Show formatted context for LLM
print("\n=== Formatted Context for LLM ===")
context = format_context_for_llm(test_results)
print(context[:1000] + "..." if len(context) > 1000 else context)

### 3.2 Custom functions

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.all_ai_applications_rating()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Shows all AI applications reveiw count, thumbs up count and average ratings'
RETURN (
  SELECT 
    `App`,
    count(*) as review_count,
    sum(`Thumbs_Up_Count`) as thumbs_up_count,
    avg(`Star_Rating`) as avg_rating
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 3 desc
)

In [0]:
%sql
select * from group4.default.all_ai_applications_rating()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_theme_category()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review theme category'
RETURN (
  SELECT 
    `Review_Theme`,
    count(*) as review_count
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 2 desc
)

In [0]:
%sql
select * from group4.default.fetch_review_theme_category()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_contents(
    theme_param STRING COMMENT 'theme category, e.g. Bugs/Performance',
    keyword_param STRING DEFAULT NULL COMMENT 'optional keyword to search in review text'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review contents by theme, with optional review text keyword'
RETURN (
    SELECT
        `App`,
        `Star_Rating`,
        `Review_Text`,
        `Word_Count`,
        `Review_Length_Chars`,
        `Thumbs_Up_Count`,
        `App_Version`,
        `Sentiment_Polarity`,
        `Review_Theme`
    FROM group4.default.ai_reviews
    WHERE `Review_Theme` = theme_param
      AND (
          keyword_param IS NULL
          OR lower(`Review_Text`) LIKE concat('%', lower(keyword_param), '%')
      )
);

In [0]:
%sql
select * 
from group4.default.fetch_review_contents('Bugs/Performance', 'crash') limit 10

### 3.3 MCP tools if used

# 4. Agent Definition

### 4.1 Tools available to the agent

### 4.2 Agent workflow / graph

In [0]:
%%writefile react_agent.py
import copy
import json
import warnings
from typing import Any, Callable, Generator, Optional
from uuid import uuid4

import backoff
import mlflow
import numpy as np
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.types import DoubleType
from sentence_transformers import SentenceTransformer
from databricks_openai import UCFunctionToolkit
from unitycatalog.ai.core.base import get_uc_function_client


############################################
# LLM endpoint — replaced by notebook cell
############################################
LLM_ENDPOINT_NAME = "__LLM_ENDPOINT__"

SYSTEM_PROMPT = """You are an AI Review Analysis Assistant specializing in generative AI application reviews.

Your role is to help users understand user feedback and sentiment about AI applications like
ChatGPT, Claude, Gemini, Midjourney, and other generative AI tools.

You have access to one tool:
- group4__default__all_ai_applications_rating: list all AI applications and their average rating.
- group4__default__fetch_review_theme_category: fetch all review theme categories.
- group4__default__fetch_review_contents: retrieve review content for a specific category, with an optional keyword filter, while limiting the number of records returned per request.


For every user question, follow this process:
1. Think about what information is needed.
2. Choose the correct tool.
3. Review the tool result.
4. If more information is needed, call another tool.
5. Then provide the final answer.

Always ground the final answer in tool results."""


############################################
# Embedding model — lazy loaded once
############################################
_embedding_model: Optional[SentenceTransformer] = None

def _get_embedding_model() -> SentenceTransformer:
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    return _embedding_model

############################################
# Cosine similarity Spark UDF
############################################
def _cosine_similarity(vec1, vec2) -> float:
    a = np.array(vec1)
    b = np.array(vec2)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

_cosine_sim_udf = udf(_cosine_similarity, DoubleType())

############################################
# Core retrieval logic
############################################
def _vector_search_exec(
    query: str,
    top_k: int = 10,
    min_similarity: float = 0.0,
) -> str:
    """
    Encode query, compute cosine similarity against ai_reviews_embeddings,
    return formatted context string ready for the LLM.
    """
    spark = SparkSession.builder.getOrCreate()
    model = _get_embedding_model()
    query_vec = model.encode(query).tolist()

    df = spark.table("group4.default.ai_reviews_embeddings")
    df = df.withColumn(
        "similarity_score",
        _cosine_sim_udf(col("embedding"), lit(query_vec)),
    )

    if min_similarity > 0:
        df = df.filter(col("similarity_score") >= min_similarity)

    results: pd.DataFrame = (
        df.orderBy(col("similarity_score").desc())
        .limit(int(top_k))
        .select(
            "App",
            "Review_Text",
            "Star_Rating",
            "Review_Date",
            "Sentiment_Polarity",
            "Review_Theme",
            "similarity_score",
        )
        .toPandas()
    )

    if results.empty:
        return "No relevant reviews found."

    parts = []
    for i, row in results.iterrows():
        parts.append(
            f"Review {i + 1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    return "\n---\n\n".join(parts)

############################################
# Tool specs (OpenAI function-calling format)
############################################
_VECTOR_SEARCH_SPEC = {
    "type": "function",
    "function": {
        "name": "vector_search",
        "description": (
            "Search reviews for qualitative topics: complaints, praise, specific user experiences. "
            "Do NOT use for aggregate statistics, rankings, or average ratings — "
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language search query describing what reviews to find",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of top results to return (default: 10)",
                    "default": 10,
                },
                "min_similarity": {
                    "type": "number",
                    "description": "Minimum similarity threshold 0-1 (default: 0.0)",
                    "default": 0.0,
                },
            },
            "required": ["query"],
        },
    },
}

############################################
# ToolInfo + agent boilerplate (unchanged)
############################################
class ToolInfo(BaseModel):
    name: str
    spec: dict
    exec_fn: Callable

    class Config:
        arbitrary_types_allowed = True

def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


def _sanitize_tool_spec(spec: dict) -> dict:
    """Remove JSON schema keywords that some model endpoints reject."""
    spec = copy.deepcopy(spec)
    params = spec.get("function", {}).get("parameters") or {}
    if not isinstance(params, dict) or "properties" not in params:
        return spec
    for prop in params.get("properties", {}).values():
        if not isinstance(prop, dict):
            continue
        t = prop.get("type")
        if t == "string":
            for key in ("minLength", "maxLength", "pattern", "format"):
                prop.pop(key, None)
        elif t in ("integer", "number"):
            for key in ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "format"):
                prop.pop(key, None)
        elif t == "array":
            for key in ("minItems", "maxItems", "uniqueItems"):
                prop.pop(key, None)
            items = prop.get("items")
            if isinstance(items, dict):
                for key in ("minLength", "maxLength", "pattern", "format",
                            "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum"):
                    items.pop(key, None)
    return spec


# ---- Register tools here ----
TOOL_INFOS: list[ToolInfo] = [
    # disable vetor search, it is so slow
    # ToolInfo(
    #     name="vector_search",
    #     spec=_VECTOR_SEARCH_SPEC,
    #     exec_fn=_vector_search_exec,
    # ),
]
# Add more ToolInfo entries here if needed
UC_TOOL_NAMES = ["group4.default.all_ai_applications_rating", 
                 "group4.default.fetch_review_theme_category", 
                 "group4.default.fetch_review_contents"
                 ]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))

############################################
# ToolCallingAgent (structure unchanged)
############################################
class ToolCallingAgent(ResponsesAgent):
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        return [_sanitize_tool_spec(t.spec) for t in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        sane_args = {k: v for k, v in (args or {}).items() if k and isinstance(k, str)}
        name = tool_name.strip().strip('"').strip("'")
        if "<" in name:
            name = name.split("<")[0].strip()
        if name in self._tools_dict:
            return self._tools_dict[name].exec_fn(**sane_args)
        candidates = [k for k in self._tools_dict if name.startswith(k)]
        if candidates:
            return self._tools_dict[max(candidates, key=len)].exec_fn(**sane_args)
        raise KeyError(f"Unknown tool: {tool_name!r}. Known tools: {list(self._tools_dict.keys())}")

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        for chunk in self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=to_chat_completions_input(messages),
            tools=self.get_tool_specs(),
            stream=True,
        ):
            chunk_dict = chunk.to_dict()
            if chunk_dict.get("choices"):
                yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))
        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 20,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role") == "assistant":
                return
            elif last_msg.get("type") == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )
        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            session_id = None
            if request.custom_inputs and "session_id" in request.custom_inputs:
                session_id = request.custom_inputs.get("session_id")
            elif request.context and request.context.conversation_id:
                session_id = request.context.conversation_id
            if session_id:
                mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
            outputs = [
                event.item
                for event in self.predict_stream(request)
                if event.type == "response.output_item.done"
            ]
            return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id
        if session_id:
            mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


mlflow.openai.autolog()

In [0]:
# Available foundation model endpoints (adjust for your workspace)
AVAILABLE_LLMS = [
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
]
# Set the LLM: by index (e.g. 0, 1, 2) or by name
CHOSEN_LLM_ENDPOINT = AVAILABLE_LLMS[0]  # change index or set CHOSEN_LLM_ENDPOINT = "your-endpoint-name"
print(f"Using LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

In [0]:
# Bake chosen LLM into agent.py (run "Choose LLM" first)
with open("react_agent.py") as f:
    content = f.read()
with open("react_agent.py", "w") as f:
    f.write(content.replace("__LLM_ENDPOINT__", CHOSEN_LLM_ENDPOINT))
print("react_agent.py updated with LLM_ENDPOINT_NAME =", CHOSEN_LLM_ENDPOINT)

# 5. Agent Test Examples

### 5.1 Normal user queries

In [0]:
import traceback
import nest_asyncio
import sys
import os

# Add the current working directory to Python path so react_agent.py can be imported
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
sys.path.insert(0, current_dir)
print(f"Added to Python path: {current_dir}")

import react_agent

nest_asyncio.apply()


AGENT = react_agent.ToolCallingAgent(llm_endpoint=react_agent.LLM_ENDPOINT_NAME, tools=react_agent.TOOL_INFOS)
import mlflow
mlflow.models.set_model(AGENT)

In [0]:
react_agent.TOOL_INFOS

In [0]:
# Because the free version of Databricks does not support Vector Search indexes, we implemented similarity search using a custom embedding model, BAAI/bge-small-en-v1.5. For each query, the data is reloaded into Spark, so it may take mintues to compelete.

AGENT.predict(
    {"input": [{"role": "user", "content": "What do users complain about app crashes?"}], "custom_inputs": {"session_id": "test-session-123"}},
)

### 5.2 Tool calls

In [0]:
# this will use tool all_ai_applications_rating to reture all AI applications rating
AGENT.predict(
    {"input": [{"role": "user", "content": "List all AI applications and their average rating"}], "custom_inputs": {"session_id": "test-session-123"}},
)

### 5.3 Final answers

# 6. Evaluation & Traces

- At least 5 traces
- LLM judge or manual evaluation
- Success/failure analysis

# 7. LLM Comparison + ROI

- Compare 2 LLMs
- Accuracy / quality difference
- Cost difference
- ROI calculation

# 8. Limitations, Deployment, and Reflection

- What worked well
- What was weak
- Human evaluation role
- Deployment recommendation
- AI tool disclosure